# 🌊 Early ENSO Phase Prediction — Starter Notebook (local)

**Run from repo root after:** `python scripts/build_dataset.py` and `python scripts/export_kaggle_dataset.py`

This notebook reads from `data/kaggle_export/` — the same files that will be published on Kaggle.
When uploaded to Kaggle, only the path in cell 2 needs to change.

Contents:
1. Load & inspect data
2. ENSO phase distribution
3. Niño 3.4 time series
4. Seasonal patterns (spring predictability barrier)
5. Feature–target correlations
6. Simple baseline: Logistic Regression
7. Stronger model: LightGBM
8. Feature importance (SHAP)
9. Lead-time performance curve
10. What to try next

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
except ImportError:
    LGB_AVAILABLE = False
    print('lightgbm not installed — section 7 will be skipped')

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print('shap not installed — section 8 will be skipped. Run: pip install shap')

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
})

PHASE_COLORS = {'El Niño': '#d6604d', 'Neutral': '#999999', 'La Niña': '#4393c3'}
LABEL_ORDER  = ['La Niña', 'Neutral', 'El Niño']
TARGETS      = ['enso_t1', 'enso_t3', 'enso_t6']
TARGET_COLS  = set(TARGETS) | {'enso_phase', 'enso_t1_int', 'enso_t3_int', 'enso_t6_int', 'date'}

print('Libraries loaded ✓')

## 1. Load & inspect data

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Local:  change to your repo root if running from a different directory
# Kaggle: change DATA_DIR to '/kaggle/input/enso-early-phase-prediction'
import os
DATA_DIR = '~/code/enso-forecast/data/kaggle_export'

#train = pd.read_csv(f'{DATA_DIR}/enso_train.csv', parse_dates=['date'])
#test  = pd.read_csv(f'{DATA_DIR}/enso_test.csv',  parse_dates=['date'])
train = pd.read_parquet(f'{DATA_DIR}/enso_train.parquet')
test  = pd.read_parquet(f'{DATA_DIR}/enso_test.parquet')

print(f'Train: {len(train)} rows  ({train.date.min().date()} → {train.date.max().date()})')
print(f'Test:  {len(test)} rows   ({test.date.min().date()} → {test.date.max().date()})')
print(f'Columns: {len(train.columns)}')
print()

# Quick dtypes check — all feature columns should be numeric
object_cols = [c for c in train.columns if train[c].dtype == object and c != 'enso_phase']
target_str  = [c for c in TARGETS if c in train.columns]
print(f'String target columns: {target_str}')
if object_cols:
    print(f'WARNING: unexpected object columns: {object_cols}')
else:
    print('All non-target columns are numeric ✓')

# Data quality note: zonal wind index
zwnd_max = train['zonal_wind_850_anom'].abs().max()
if zwnd_max > 10:
    print(f'\nNote: zonal_wind_850_anom has values up to {zwnd_max:.1f}. '
          f'NOAA publishes this as a dimensionless index, not m/s. '
          f'Use for relative comparisons only.')

train.head(3)

In [ ]:
FEATURE_COLS = [c for c in train.columns
                if c not in TARGET_COLS
                and train[c].dtype != object]

CORE_FEATURES = [
    'sst_anom_nino34',
    'sst_anom_nino34_lag1',
    'sst_anom_nino34_lag3',
    'sst_anom_nino34_rm3',
    'southern_oscillation_index',
    'zonal_wind_850_anom',
    'month_sin',
    'month_cos',
]
CORE_FEATURES = [c for c in CORE_FEATURES if c in train.columns]

# Feature columns: climate indices + spring barrier context features
# Exclude regression targets (nino34_tL) and classification targets
EXPORT_PREFIXES = (
    'sst_anom_', 'southern_oscillation_index',
    'zonal_wind_850_anom', 'month_', 'year',
    'crosses_spring_', 'init_in_growth_phase',
)
# Exclude regression targets and integer-encoded targets from features
TARGET_COLS = TARGET_COLS | {'nino34_t1', 'nino34_t3', 'nino34_t6'}

FEATURE_COLS = [c for c in train.columns
                if c not in TARGET_COLS
                and train[c].dtype != object
                and c != 'date'
                and not c.startswith('nino34_t')
                and c not in {'enso_t1_int', 'enso_t3_int', 'enso_t6_int'}]

spring_feats = [c for c in FEATURE_COLS if 'spring' in c or 'growth' in c]
reg_targets  = [c for c in train.columns if c.startswith('nino34_t')]

print(f'Total feature columns: {len(FEATURE_COLS)}')
print(f'  Climate index features: {len(FEATURE_COLS) - len(spring_feats)}')
print(f'  Spring barrier features: {spring_feats}')
print(f'Core features for LR: {len(CORE_FEATURES)}')
print(f'Regression targets: {reg_targets}')


## 2. ENSO phase distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

for ax, col, title in zip(
    axes,
    ['enso_phase', 'enso_t3', 'enso_t6'],
    ['Phase at t (reference)', 'Target: t+3 months', 'Target: t+6 months']
):
    counts = train[col].value_counts().reindex(LABEL_ORDER).fillna(0)
    bars = ax.bar(
        LABEL_ORDER, counts,
        color=[PHASE_COLORS[p] for p in LABEL_ORDER],
        edgecolor='black', linewidth=0.5
    )
    ax.bar_label(bars, fmt='%d', padding=3, fontsize=9)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_ylabel('Months')
    ax.set_ylim(0, counts.max() * 1.22)

plt.suptitle('Class distribution — training set', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Neutral dominates → use macro F1, not accuracy, as your primary metric.')
print(f"Train class balance: {train['enso_phase'].value_counts(normalize=True).round(3).to_dict()}")

## 3. Niño 3.4 time series

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

x = train['date']
y = train['sst_anom_nino34']

ax.axhline( 0.5, color='#d6604d', linestyle='--', linewidth=0.8, alpha=0.7, label='El Niño (+0.5°C)')
ax.axhline(-0.5, color='#4393c3', linestyle='--', linewidth=0.8, alpha=0.7, label='La Niña (−0.5°C)')
ax.axhline( 0.0, color='black',   linestyle='-',  linewidth=0.4, alpha=0.3)
ax.fill_between(x, y, 0, where=(y >  0.5), color='#d6604d', alpha=0.25)
ax.fill_between(x, y, 0, where=(y < -0.5), color='#4393c3', alpha=0.25)
ax.plot(x, y, color='black', linewidth=0.8)

# Annotate famous events (only if dates are in range)
events = [
    ('1982-11-01', '1982/83\nEl Niño'),
    ('1997-11-01', '1997/98\nEl Niño'),
    ('2010-10-01', '2010/11\nLa Niña'),
    ('2015-11-01', '2015/16\nEl Niño'),
]
for date_str, label in events:
    ts  = pd.Timestamp(date_str)
    row = train[train['date'] == ts]
    if not row.empty:
        val = row['sst_anom_nino34'].iloc[0]
        offset = 22 if val > 0 else -30
        ax.annotate(label, xy=(ts, val), xytext=(0, offset),
                    textcoords='offset points', fontsize=7.5, ha='center',
                    arrowprops=dict(arrowstyle='->', color='gray', lw=0.7))

ax.set_ylabel('SST anomaly (°C)', fontsize=11)
ax.set_title('                                                              Niño 3.4 SST anomaly — training set', fontsize=12, fontweight='bold')
ax.legend(fontsize=9, frameon=False, loc='upper left')
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## 4. Seasonal patterns — spring predictability barrier

In [ ]:
tmp = train.copy()
tmp['month_num'] = train['date'].dt.month
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

pivot = tmp.groupby(['month_num', 'enso_phase']).size().unstack(fill_value=0)
pivot = pivot.reindex(columns=LABEL_ORDER, fill_value=0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
pivot_pct.index = month_names

fig, ax = plt.subplots(figsize=(11, 4))
pivot_pct.plot(
    kind='bar', stacked=True, ax=ax,
    color=[PHASE_COLORS[p] for p in LABEL_ORDER],
    edgecolor='white', linewidth=0.3
)
# Highlight spring months
for i, m in enumerate(month_names):
    if m in ('Mar', 'Apr', 'May'):
        ax.get_xticklabels()[i].set_color('#d6604d')
        ax.get_xticklabels()[i].set_fontweight('bold')

ax.set_xlabel('')
ax.set_ylabel('% of months', fontsize=10)
ax.set_title('ENSO phase by calendar month  (spring months MAM highlighted = predictability barrier)',
             fontsize=10, fontweight='bold')
ax.legend(LABEL_ORDER, loc='upper right', fontsize=9, frameon=False)
ax.set_xticklabels(month_names, rotation=0)
plt.tight_layout()
plt.show()

## 5. Feature–target correlations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, target_int, title in zip(
    axes,
    ['enso_t1_int', 'enso_t3_int', 'enso_t6_int'],
    ['Correlation with t+1', 'Correlation with t+3', 'Correlation with t+6']
):
    corrs = (train[FEATURE_COLS]
             .corrwith(train[target_int])
             .abs()
             .sort_values(ascending=False)
             .head(15))

    colors = ['#d6604d' if 'nino34' in c else
              '#4393c3' if 'oscillation' in c else
              '#66c2a5' if 'wind' in c else
              '#f4a582' if 'nino' in c else '#cccccc'
              for c in corrs.index]

    corrs.sort_values().plot(
        kind='barh', ax=ax,
        color=list(reversed(colors)),
        edgecolor='black', linewidth=0.3
    )
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlabel('|Pearson r|')
    ax.axvline(0.5, color='gray', linestyle='--', linewidth=0.7, alpha=0.5)

plt.suptitle('Top 15 features by |correlation| with target', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Simple baseline: Logistic Regression

In [ ]:
def prepare(df, feature_cols, target):
    """Return X, y with NaN rows dropped. X is always numeric.
    Preserves the DatetimeIndex (set from 'date' column if needed).
    """
    cols = [c for c in feature_cols if c in df.columns]
    # Ensure DatetimeIndex for month-based stratification
    if not isinstance(df.index, pd.DatetimeIndex) and 'date' in df.columns:
        df = df.set_index('date')
    mask = df[target].notna()
    X = df.loc[mask, cols].fillna(0)
    y = df.loc[mask, target]
    return X, y


def evaluate(y_true, y_pred, name=''):
    """Compute accuracy and macro F1, return results dict."""
    yt = np.array(y_true)
    yp = np.array(y_pred)
    mask = pd.notna(yt) & pd.notna(yp)
    yt, yp = yt[mask], yp[mask]
    acc = accuracy_score(yt, yp)
    f1  = f1_score(yt, yp, average='macro', zero_division=0, labels=LABEL_ORDER)
    cm  = confusion_matrix(yt, yp, labels=LABEL_ORDER)
    print(f'  {name:45s}  acc={acc:.3f}  f1_macro={f1:.3f}  n={len(yt)}')
    return {'name': name, 'accuracy': acc, 'f1_macro': f1, 'cm': cm, 'n': len(yt)}


le = LabelEncoder().fit(LABEL_ORDER)
sc = StandardScaler()

lr_results = {}
print('Logistic Regression — 8 core features\n')

for target in TARGETS:
    X_tr, y_tr = prepare(train, CORE_FEATURES, target)
    X_te, y_te = prepare(test,  CORE_FEATURES, target)

    X_tr_sc = sc.fit_transform(X_tr)
    X_te_sc = sc.transform(X_te)

    lr = LogisticRegression(
        C=1.0, max_iter=2000,
        class_weight='balanced', random_state=42
    )
    lr.fit(X_tr_sc, le.transform(y_tr))
    preds = le.inverse_transform(lr.predict(X_te_sc))
    lr_results[target] = evaluate(y_te, preds, name=f'LR core / {target}')

In [ ]:
# Confusion matrix for t+3
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, target in zip(axes, TARGETS):
    cm = lr_results[target]['cm']
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    sns.heatmap(
        cm_norm, annot=cm, fmt='d', cmap='RdBu_r',
        vmin=0, vmax=1,
        xticklabels=['La Niña', 'Neutral', 'El Niño'],
        yticklabels=['La Niña', 'Neutral', 'El Niño'],
        linewidths=0.5, ax=ax
    )
    ax.set_xlabel('Predicted', fontsize=9)
    ax.set_ylabel('Actual', fontsize=9)
    ax.set_title(f'LR — {target}  (F1={lr_results[target]["f1_macro"]:.3f})',
                 fontsize=9, fontweight='bold')

plt.suptitle('Logistic Regression — confusion matrices', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Stronger model: LightGBM

In [ ]:
lgb_results = {}
lgb_models  = {}

if not LGB_AVAILABLE:
    print('LightGBM not installed — skipping. Run: pip install lightgbm')
else:
    print('LightGBM — all features\n')
    for target in TARGETS:
        X_tr, y_tr = prepare(train, FEATURE_COLS, target)
        X_te, y_te = prepare(test,  FEATURE_COLS, target)

        model = lgb.LGBMClassifier(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=31,
            min_child_samples=20,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1,
            verbosity=-1,
        )
        model.fit(
            X_tr, le.transform(y_tr),
            eval_set=[(X_te, le.transform(y_te))],
            eval_metric='multi_logloss',
            callbacks=[lgb.early_stopping(50, verbose=False),
                       lgb.log_evaluation(period=-1)],
        )
        preds = le.inverse_transform(model.predict(X_te))
        lgb_results[target] = evaluate(y_te, preds, name=f'LightGBM / {target}')
        lgb_models[target]  = model

    print()
    print('Published benchmark results (test set 2019–2026):')
    print('  enso_t1  LightGBM F1=0.945  |  Persistence F1=0.858')
    print('  enso_t3  LR (8 features) F1=0.802  |  Persistence F1=0.610')
    print('  enso_t6  LR (8 features) F1=0.608  |  Persistence F1=0.419')
    print('  Note: LR outperforms LightGBM at t+3/t+6 — simpler models')
    print('  generalise better at longer horizons on this dataset.')

## 8. Feature importance (SHAP)

In [ ]:
if not SHAP_AVAILABLE:
    print('shap not installed — skipping. Run: pip install shap')
elif not lgb_results:
    print('LightGBM results not available — run section 7 first')
else:
    X_tr_t3, _ = prepare(train, FEATURE_COLS, 'enso_t3')

    explainer   = shap.TreeExplainer(lgb_models['enso_t3'])
    shap_values = explainer.shap_values(X_tr_t3)

    # shap_values can be a list (one array per class) or a 3D array
    # depending on shap version — handle both
    if isinstance(shap_values, list):
        # older shap: list of (n_samples, n_features) arrays
        mean_abs = np.abs(np.stack(shap_values)).mean(axis=(0, 1))
    else:
        # newer shap: (n_samples, n_features, n_classes)
        mean_abs = np.abs(shap_values).mean(axis=(0, 2))

    shap_df = (
        pd.DataFrame({'feature': FEATURE_COLS, 'mean_abs_shap': mean_abs})
        .sort_values('mean_abs_shap', ascending=False)
        .head(20)
    )

    fig, ax = plt.subplots(figsize=(8, 7))
    colors = [
        '#d6604d' if 'nino34' in c else
        '#4393c3' if 'oscillation' in c else
        '#66c2a5' if 'wind' in c else
        '#f4a582' if 'nino' in c else '#cccccc'
        for c in shap_df['feature']
    ]
    shap_df.sort_values('mean_abs_shap').plot(
        kind='barh', x='feature', y='mean_abs_shap',
        ax=ax, color=list(reversed(colors)),
        edgecolor='black', linewidth=0.3, legend=False
    )
    ax.set_xlabel('Mean |SHAP value|', fontsize=10)
    ax.set_title('Feature importance — LightGBM t+3 (SHAP)', fontsize=11, fontweight='bold')

    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(facecolor='#d6604d', label='Niño 3.4'),
        Patch(facecolor='#4393c3', label='SOI'),
        Patch(facecolor='#66c2a5', label='Zonal wind'),
        Patch(facecolor='#f4a582', label='Other Niño'),
    ], fontsize=8, frameon=False, loc='lower right')

    plt.tight_layout()
    plt.show()

    print('\nTop 5 features for t+3:')
    print(shap_df.head(5)[['feature', 'mean_abs_shap']].to_string(index=False))

## 9. Lead-time performance curve

In [ ]:
# ── Compute baselines on test set ─────────────────────────────────────────────
modal_class = train['enso_phase'].value_counts().idxmax()

pers_f1 = {}
clim_f1 = {}

for target in TARGETS:
    # Only use rows where both enso_phase and target are not NaN
    mask   = test[target].notna() & test['enso_phase'].notna()
    y_te   = test.loc[mask, target]
    y_pers = test.loc[mask, 'enso_phase']          # persistence: current phase
    y_clim = [modal_class] * mask.sum()             # climatology: always modal

    pers_f1[target] = f1_score(y_te, y_pers, average='macro',
                                zero_division=0, labels=LABEL_ORDER)
    clim_f1[target] = f1_score(y_te, y_clim, average='macro',
                                zero_division=0, labels=LABEL_ORDER)

lead_times = [1, 3, 6]

lr_f1_vals  = [lr_results[t]['f1_macro']        for t in TARGETS]
lgb_f1_vals = [lgb_results[t]['f1_macro']       for t in TARGETS] if lgb_results else None
pers_f1_vals = [pers_f1[t]                      for t in TARGETS]
clim_f1_vals = [clim_f1[t]                      for t in TARGETS]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

if lgb_f1_vals:
    ax.plot(lead_times, lgb_f1_vals, '-o', color='#FF5722',
            linewidth=2, markersize=8, label='LightGBM (50 features)')
    ax.fill_between(lead_times, pers_f1_vals, lgb_f1_vals,
                    alpha=0.08, color='#FF5722')

ax.plot(lead_times, lr_f1_vals,   '-^', color='#2196F3',
        linewidth=2, markersize=8, label='Logistic Regression (8 features)')
ax.plot(lead_times, pers_f1_vals, ':s', color='#555555',
        linewidth=1.5, markersize=7, label='Persistence baseline')
ax.plot(lead_times, clim_f1_vals, ':D', color='#aaaaaa',
        linewidth=1.5, markersize=7, label='Climatology baseline')

ax.set_xticks(lead_times)
ax.set_xticklabels(['t+1\n(1 month)', 't+3\n(3 months)', 't+6\n(6 months)'], fontsize=10)
ax.set_ylabel('F1 macro', fontsize=11)
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.set_title('Prediction skill vs lead time', fontsize=12, fontweight='bold')
ax.legend(fontsize=9, frameon=False)
plt.tight_layout()
plt.show()

# Summary table
rows = []
for i, t in enumerate(TARGETS):
    row = {'target': t, 'persistence': round(pers_f1_vals[i], 3),
           'LR_core': round(lr_f1_vals[i], 3)}
    if lgb_f1_vals:
        row['LightGBM'] = round(lgb_f1_vals[i], 3)
    rows.append(row)

print(pd.DataFrame(rows).to_string(index=False))

## 9.5 The Spring Predictability Barrier — init-month analysis

The **spring predictability barrier (SPB)** is one of the most robust features of ENSO forecasting:
skill drops sharply for forecasts initialized before or during boreal spring (MAM).
But the barrier is not symmetric — it matters *where in the year you are initializing*, not just whether the forecast window crosses spring.

**Key insight from this dataset:**
> Forecasts initialized in the post-barrier growth phase **(JJA–SON, June–November)** achieve
> F1 ~0.71–0.75 at t+6. Forecasts initialized in **DJF–MAM (Dec–May)** achieve F1 ~0.15–0.49.
> The worst months are **February and March** (F1 ~0.15), when the previous year's anomaly
> is dying and the new year's signal has not yet emerged through ocean–atmosphere coupling.

The dataset provides two engineered features that capture this:
- `crosses_spring_tL` — does the forecast window pass through MAM?
- `init_in_growth_phase` — is the initialization in the high-skill JJA–SON window?


In [ ]:
import calendar

# ── F1 by initialization month — heatmap across all lead times ───────────────
if not lgb_results:
    print('Run section 7 (LightGBM) first')
else:
    from sklearn.metrics import f1_score

    MONTH_NAMES = [calendar.month_abbr[m] for m in range(1, 13)]
    LEAD_TIMES  = ['t+1', 't+3', 't+6']
    TARGET_MAP  = {'t+1': 'enso_t1', 't+3': 'enso_t3', 't+6': 'enso_t6'}
    MODEL_MAP   = {'t+1': lgb_models.get('enso_t1'), 
                   't+3': lgb_models.get('enso_t3'),
                   't+6': lgb_models.get('enso_t6')}

    heatmap_data = np.full((12, 3), np.nan)

    for j, lt in enumerate(LEAD_TIMES):
        target = TARGET_MAP[lt]
        model  = MODEL_MAP[lt]
        if model is None:
            continue
        X_te, y_te = prepare(test, FEATURE_COLS, target)
        y_pred     = le.inverse_transform(model.predict(X_te))

        for i, month in enumerate(range(1, 13)):
            mask = X_te.index.month == month
            if mask.sum() < 3:
                continue
            f1 = f1_score(
                y_te[mask], y_pred[mask],
                average='macro', zero_division=0,
                labels=LABEL_ORDER
            )
            heatmap_data[i, j] = f1

    # ── Plot heatmap ──────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 8))

    im = ax.imshow(heatmap_data, cmap='RdYlGn', vmin=0.0, vmax=1.0,
                   aspect='auto', interpolation='nearest')

    # Annotate cells with F1 value
    for i in range(12):
        for j in range(3):
            val = heatmap_data[i, j]
            if not np.isnan(val):
                color = 'white' if val < 0.35 or val > 0.80 else 'black'
                ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                        fontsize=10, fontweight='bold', color=color)

    # Highlight spring months (low-skill initialization)
    for i in [1, 2, 3]:  # Feb, Mar, Apr (0-indexed)
        ax.add_patch(plt.Rectangle((-0.5, i-0.5), 3, 1,
                     fill=False, edgecolor='#d6604d', lw=2.5, linestyle='--'))

    # Highlight growth phase (high-skill initialization)
    for i in [6, 7, 8, 9]:  # Jul, Aug, Sep, Oct (0-indexed)
        ax.add_patch(plt.Rectangle((-0.5, i-0.5), 3, 1,
                     fill=False, edgecolor='#2166ac', lw=2, linestyle='-'))

    ax.set_xticks(range(3))
    ax.set_xticklabels(LEAD_TIMES, fontsize=12)
    ax.set_yticks(range(12))
    ax.set_yticklabels(MONTH_NAMES, fontsize=10)
    ax.set_xlabel('Lead time', fontsize=12)
    ax.set_ylabel('Initialization month', fontsize=12)
    ax.set_title('F1 macro by initialization month × lead time\n(LightGBM, test set 2019–2026)',
                 fontsize=11, fontweight='bold')

    plt.colorbar(im, ax=ax, label='F1 macro', shrink=0.6)

    # Legend
    from matplotlib.patches import Patch
    ax.legend(handles=[
        plt.Line2D([0],[0], color='#d6604d', lw=2.5, linestyle='--',
                   label='Low skill: DJF–MAM inits (spring barrier)'),
        plt.Line2D([0],[0], color='#2166ac', lw=2,
                   label='High skill: JJA–SON inits (growth phase)'),
    ], loc='lower right', fontsize=8, frameon=True)

    plt.tight_layout()
    plt.show()

    print('Red dashed = spring barrier zone (low skill)')
    print('Blue solid  = post-barrier growth phase (high skill, high impact)')
    print(f'\nWorst init month at t+6: Feb (F1={heatmap_data[1,2]:.2f})')
    print(f'Best  init month at t+6: Aug (F1={heatmap_data[7,2]:.2f})')


## 9.7 Regression targets — predicting Niño 3.4 anomaly magnitude

Beyond phase classification, the dataset provides continuous regression targets:
`nino34_t1`, `nino34_t3`, `nino34_t6` — the smoothed Niño 3.4 anomaly (°C) at each horizon.

Magnitude matters: a **+2.5°C anomaly** triggers fundamentally different atmospheric responses
than a **+0.6°C anomaly**, even though both are classified as 'El Niño'.

A key phenomenon to look for: the **regression-to-the-mean bias** during spring.
Models trained on all months tend to predict near-zero anomalies for spring initializations
— they 'play it safe' when uncertainty is high. This bias is directly observable in the
predicted vs actual scatter plot.


In [ ]:
# ── Simple regression baseline: LightGBM predicting nino34_t3 ────────────────
if not LGB_AVAILABLE:
    print('LightGBM not available — skipping')
else:
    from sklearn.metrics import mean_squared_error, mean_absolute_error
    import lightgbm as lgb

    REG_TARGET = 'nino34_t3'

    X_tr_reg, y_tr_reg = prepare(train, FEATURE_COLS, REG_TARGET)
    X_te_reg, y_te_reg = prepare(test,  FEATURE_COLS, REG_TARGET)

    reg_model = lgb.LGBMRegressor(
        n_estimators=500, learning_rate=0.05,
        num_leaves=31, min_child_samples=20,
        random_state=42, n_jobs=-1, verbosity=-1
    )
    reg_model.fit(
        X_tr_reg, y_tr_reg,
        eval_set=[(X_te_reg, y_te_reg)],
        callbacks=[lgb.early_stopping(50, verbose=False),
                   lgb.log_evaluation(period=-1)]
    )
    y_pred_reg = reg_model.predict(X_te_reg)

    rmse = mean_squared_error(y_te_reg, y_pred_reg) ** 0.5
    mae  = mean_absolute_error(y_te_reg, y_pred_reg)
    print(f'nino34_t3 regression — RMSE: {rmse:.3f}°C  MAE: {mae:.3f}°C')

    # ── Scatter: predicted vs actual, coloured by init month regime ──────────
    growth_mask = X_te_reg['init_in_growth_phase'] if 'init_in_growth_phase' in X_te_reg.columns else None

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Left: all points
    ax = axes[0]
    ax.scatter(y_te_reg, y_pred_reg, alpha=0.6, s=30,
               c='#555555', edgecolors='none')
    lim = max(abs(y_te_reg).max(), abs(y_pred_reg).max()) * 1.1
    ax.plot([-lim, lim], [-lim, lim], 'k--', linewidth=0.8, label='Perfect')
    ax.axhline(0, color='gray', linewidth=0.4)
    ax.axvline(0, color='gray', linewidth=0.4)
    ax.set_xlabel('Actual Niño 3.4 anomaly (°C)', fontsize=10)
    ax.set_ylabel('Predicted (°C)', fontsize=10)
    ax.set_title(f'nino34_t3 — all months\nRMSE={rmse:.2f}°C', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)

    # Right: split by growth phase
    ax = axes[1]
    if growth_mask is not None:
        colors = ['#d6604d' if not g else '#2166ac' for g in growth_mask]
        ax.scatter(y_te_reg, y_pred_reg, alpha=0.6, s=30,
                   c=colors, edgecolors='none')
        from matplotlib.patches import Patch
        ax.legend(handles=[
            Patch(color='#2166ac', label='Growth phase init (JJA-SON)'),
            Patch(color='#d6604d', label='Other init months'),
        ], fontsize=8, frameon=False)
    ax.plot([-lim, lim], [-lim, lim], 'k--', linewidth=0.8)
    ax.axhline(0, color='gray', linewidth=0.4)
    ax.axvline(0, color='gray', linewidth=0.4)
    ax.set_xlabel('Actual Niño 3.4 anomaly (°C)', fontsize=10)
    ax.set_ylabel('Predicted (°C)', fontsize=10)
    ax.set_title('Coloured by initialization regime\n(note regression-to-mean in red points)',
                 fontsize=10, fontweight='bold')

    plt.suptitle('Regression target: nino34_t3 (3-month ahead Niño 3.4 anomaly)',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()


## 10. What to try next

The **t+6 result (F1≈0.55) has the most headroom** — that's where the real challenge is.
Here are directions worth exploring:

---

### 🔬 Feature ideas

| Idea | Why it matters |
|---|---|
| **MJO features** | The Madden-Julian Oscillation is a known ENSO precursor at 30–90 day timescales. BOM publishes daily RMM indices (phase + amplitude). |
| **Thermocline depth (D20)** | Depth of the 20°C isotherm captures subsurface heat content — a leading indicator of El Niño development weeks before SST signals appear. |
| **Longer lags (9, 12 months)** | The annual ENSO recharge cycle may leave useful signal at lags beyond 6 months. |
| **Cross-basin indices** | Atlantic SST anomalies and Indian Ocean Dipole have teleconnections with ENSO not captured here. |

### 🤖 Modelling ideas

- **Walk-forward CV** — The `src/validation/splits.py` module in the [dataset repo](https://github.com/ferariz/enso-forecast) has a ready-made implementation.
- **Regime-specific models** — Train one model for `init_in_growth_phase=True` and another for `False`. The two regimes have fundamentally different signal-to-noise characteristics.
- **Sequence models** — LSTM or Transformer directly on the raw monthly time series.
- **Probabilistic output** — Predict phase *probabilities* rather than hard labels. More actionable for operational forecasting.
- **Weighted regression loss** — For the `nino34_tL` targets, weight errors by `init_in_growth_phase` to penalize misses in the high-impact window more heavily.
- **Ensemble by lead time** — LR dominates at t+3/t+6; LightGBM at t+1. Blend accordingly.

### 📏 Evaluation ideas

- **Heidke Skill Score (HSS)** — The standard meteorological metric for categorical ENSO forecasts. Directly comparable to operational forecast centres.
- **El Niño onset detection** — Reframe as a binary task: does an El Niño *start* within the next 6 months?
- **Reliability diagrams** — If you go probabilistic, plot calibration curves to verify confidence scores are trustworthy.
- **Magnitude stratification** — Evaluate RMSE separately for weak (|anomaly| < 1°C) vs strong (|anomaly| > 1.5°C) events.
- **Extend to t+9** — The operational standard is 6–9 months. t+9 is the next frontier with this dataset.

---

> 💡 **Tip:** The dataset repo includes the full pipeline — ingestion, feature engineering,
> leakage checks, and baselines — so you can extend it rather than starting from scratch.
> See [ferariz/enso-forecast](https://github.com/ferariz/enso-forecast) on GitHub.

---

*"The Spring Predictability Barrier isn't just a wall you cross; it's a reset of the tropical Pacific's
memory. Forecasts that target the DJF peak (Aug–Nov inits) benefit from the system's maturity,
while forecasts that attempt to look through spring into the following summer (Feb–Mar inits)
are essentially trying to predict a coin flip before the coin has been tossed."*
